#Pelatihan & Evaluasi Model Bi-LSTM

In [5]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Import fungsi clean_text dari modul buatanmu
from preprocessing import clean_text

# 1. Load dataset hasil split
df_train = pd.read_csv("train.csv")
df_val = pd.read_csv("val.csv")
df_in_test = pd.read_csv("test_indomain.csv")
df_out_test = pd.read_csv("test_outdomain.csv")

# 2. Terapkan preprocessing pada semua split
for df in [df_train, df_val, df_in_test, df_out_test]:
    df["clean_text"] = df["text"].astype(str).apply(clean_text)

# Hapus baris yang menjadi kosong setelah dibersihkan (misal komentar hanya berisi emoji/link)
df_train = df_train[df_train["clean_text"] != ""].reset_index(drop=True)
df_val = df_val[df_val["clean_text"] != ""].reset_index(drop=True)
df_in_test = df_in_test[df_in_test["clean_text"] != ""].reset_index(drop=True)
df_out_test = df_out_test[df_out_test["clean_text"] != ""].reset_index(drop=True)

print(f"Data siap latih -> Train: {len(df_train)}, Val: {len(df_val)}")

Data siap latih -> Train: 27224, Val: 3404


##Tokenization, Label Encoding & Class Weight

In [6]:
MAX_VOCAB = 15000  # Kapasitas kosakata
MAX_LEN = 100      # Panjang maksimal kata per komentar

# 1. Fit Tokenizer HANYA pada data train
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df_train["clean_text"])

X_train = pad_sequences(tokenizer.texts_to_sequences(df_train["clean_text"]), maxlen=MAX_LEN, padding="post", truncating="post")
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val["clean_text"]), maxlen=MAX_LEN, padding="post", truncating="post")
X_in_test = pad_sequences(tokenizer.texts_to_sequences(df_in_test["clean_text"]), maxlen=MAX_LEN, padding="post", truncating="post")
X_out_test = pad_sequences(tokenizer.texts_to_sequences(df_out_test["clean_text"]), maxlen=MAX_LEN, padding="post", truncating="post")

# 2. Encode label (negative=0, neutral=1, positive=2)
le = LabelEncoder()
y_train = le.fit_transform(df_train["label"])
y_val = le.transform(df_val["label"])
y_in_test = le.transform(df_in_test["label"])
y_out_test = le.transform(df_out_test["label"])

print("Mapping Kelas:", dict(zip(le.classes_, le.transform(le.classes_))))

# 3. Hitung Class Weight untuk mengatasi imbalance
weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(weights))
print("Class Weights:", class_weight_dict)

Mapping Kelas: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}
Class Weights: {0: np.float64(0.5449268400088072), 1: np.float64(3.4192413966340114), 2: np.float64(1.146225422087491)}


##Unduh Vektor Kata FastText Indonesia & Buat Embedding Matrix

In [7]:
# Download & ekstrak FastText Indonesia (300 dimensi)
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz
!gunzip -f cc.id.300.vec.gz

EMBEDDING_DIM = 300
word_index = tokenizer.word_index
num_words = min(MAX_VOCAB, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))

hits = 0
with open("cc.id.300.vec", "r", encoding="utf-8", errors="ignore") as f:
    next(f)  # Lewati baris header pertama
    for line in f:
        values = line.rstrip().split(" ")
        word = values[0]
        if word in word_index and word_index[word] < MAX_VOCAB:
            embedding_matrix[word_index[word]] = np.asarray(values[1:], dtype="float32")
            hits += 1

print(f"Berhasil mencocokkan {hits} dari {num_words} kata dengan FastText!")

Berhasil mencocokkan 14258 dari 15000 kata dengan FastText!


##Bangun & Latih Model Bi-LSTM

In [8]:
os.makedirs("models", exist_ok=True)

model = Sequential([
    Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True  # Fine-tune vektor agar menyesuaikan konteks YouTube
    ),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Bidirectional(LSTM(32, dropout=0.2)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ModelCheckpoint("models/best_bilstm_model.keras", monitor="val_loss", save_best_only=True, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/15
426/426 ━━━━━━━━━━━━━━━━━━━━ 0s 661ms/step - accuracy: 0.5965 - loss: 0.7660
Epoch 1: val_loss improved from None to 0.50166, saving model to models/best_bilstm_model.keras

Epoch 1: finished saving model to models/best_bilstm_model.keras
426/426 ━━━━━━━━━━━━━━━━━━━━ 306s 684ms/step - accuracy: 0.6967 - loss: 0.6519 - val_accuracy: 0.7905 - val_loss: 0.5017 - learning_rate: 0.0010
Epoch 2/15
426/426 ━━━━━━━━━━━━━━━━━━━━ 0s 639ms/step - accuracy: 0.8260 - loss: 0.4280
Epoch 2: val_loss did not improve from 0.50166
426/426 ━━━━━━━━━━━━━━━━━━━━ 305s 655ms/step - accuracy: 0.8232 - loss: 0.4310 - val_accuracy: 0.7952 - val_loss: 0.5022 - learning_rate: 0.0010
Epoch 3/15
426/426 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - accuracy: 0.8718 - loss: 0.3228
Epoch 3: val_loss improved from 0.50166 to 0.48658, saving model to models/best_bilstm_model.keras

Epoch 3: finished saving model to models/best_bilstm_model.keras
426/426 ━━━━━━━━━━━━━━━━━━━━ 284s 667ms/step - accuracy: 0.8655 - loss: 

##Evaluasi Lintas Domain & Simpan Artefak Model

In [9]:
# 1. Evaluasi In-Domain (Campuran Karhutla + Umum)
y_pred_in = np.argmax(model.predict(X_in_test), axis=1)
print("=== EVALUASI IN-DOMAIN TEST ===")
print(classification_report(y_in_test, y_pred_in, target_names=le.classes_))

# 2. Evaluasi Out-of-Domain (Murni Lintas Topik / Non-Karhutla)
y_pred_out = np.argmax(model.predict(X_out_test), axis=1)
print("=== EVALUASI OUT-OF-DOMAIN TEST (LINTAS TOPIK) ===")
print(classification_report(y_out_test, y_pred_out, target_names=le.classes_))

# 3. Simpan Tokenizer dan LabelEncoder untuk Streamlit
with open("models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("models/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Semua artefak model berhasil disimpan di folder models/!")

107/107 ━━━━━━━━━━━━━━━━━━━━ 23s 209ms/step
=== EVALUASI IN-DOMAIN TEST ===
              precision    recall  f1-score   support

    negative       0.90      0.79      0.84      2084
     neutral       0.43      0.72      0.54       333
    positive       0.81      0.84      0.82       990

    accuracy                           0.80      3407
   macro avg       0.71      0.78      0.74      3407
weighted avg       0.83      0.80      0.81      3407

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step
=== EVALUASI OUT-OF-DOMAIN TEST (LINTAS TOPIK) ===
              precision    recall  f1-score   support

    negative       0.89      0.80      0.84       156
     neutral       0.84      0.83      0.83        52
    positive       0.89      0.95      0.92       292

    accuracy                           0.89       500
   macro avg       0.88      0.86      0.87       500
weighted avg       0.89      0.89      0.89       500

Semua artefak model berhasil disimpan di folder models/!
